In [1]:
%%writefile ../src/features.py
"""Which columns may be used to predict lateness. Pipelines are added later."""
from src import config

# Known only after the shipment happens, or computed from the outcome itself
OUTCOME_COLS = [
    config.REAL_DAYS, config.GAP, config.DELIVERY_STATUS,
    config.DELIVERY_CLASS, config.IS_CANCELED,
]

# Not known at order time, or net of costs that arise during fulfilment.
# Excluded to be safe; can be revisited if the business confirms timing.
EXCLUDED_UNSURE_COLS = [
    "Order Status",                      # changes after the order is placed
    "Benefit per order", "Order Profit Per Order", "Order Item Profit Ratio",
]

# Duplicates of other information
REDUNDANT_COLS = [
    config.SCHED_DAYS,                   # fully determined by Shipping Mode
    "Category Id", "Department Id",      # same as the name columns
    "Sales per customer",                # equals Order Item Total in this file
]

# Identifiers (a model should not memorise individuals)
ID_COLS = ["Customer Id"]

NEVER_USE = [config.TARGET] + OUTCOME_COLS + EXCLUDED_UNSURE_COLS + REDUNDANT_COLS + ID_COLS


def candidate_features(df):
    """Columns left after removing everything above."""
    return [c for c in df.columns if c not in NEVER_USE]

Overwriting ../src/features.py


In [2]:
%%writefile ../src/split.py
"""Train/test split. The test set is created once and not used until the end."""
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from src import config

GROUP_COL = "Customer Id"


def make_split(d: pd.DataFrame, test_fraction: float = 0.2):
    """Stratified on the target, grouped by customer.

    Grouped: no customer appears in both train and test, so the test score
    is not inflated by seeing the same customer twice.
    """
    n_splits = round(1 / test_fraction)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=config.RANDOM_STATE)
    train_idx, test_idx = next(sgkf.split(d, d[config.TARGET], groups=d[GROUP_COL]))
    return d.iloc[train_idx].reset_index(drop=True), d.iloc[test_idx].reset_index(drop=True)

Writing ../src/split.py


In [3]:
import sys
sys.path.append("..")
import pandas as pd
from src import config
from src.data import delivered_only
from src.split import make_split, GROUP_COL
from src.features import candidate_features, NEVER_USE

d = delivered_only(pd.read_parquet(config.PROCESSED_PATH)).reset_index(drop=True)
train, test = make_split(d)

print("All      :", d.shape)
print("Train    :", train.shape, " late rate:", round(train[config.TARGET].mean(), 4))
print("Test     :", test.shape,  " late rate:", round(test[config.TARGET].mean(), 4))
print("Customers in both train and test:",
      len(set(train[GROUP_COL]) & set(test[GROUP_COL])))

train.to_parquet(config.ROOT / "data/processed/train.parquet", index=False)
test.to_parquet(config.ROOT / "data/processed/test.parquet", index=False)

All      : (172765, 38)
Train    : (138213, 38)  late rate: 0.5729
Test     : (34552, 38)  late rate: 0.5729
Customers in both train and test: 0


In [4]:
feats = candidate_features(train)
print(len(feats), "candidate features:")
print(feats)
print("\nShipping mix, train vs test:")
print(pd.concat([train[config.SHIP_MODE].value_counts(normalize=True),
                 test[config.SHIP_MODE].value_counts(normalize=True)],
                axis=1, keys=["train", "test"]).round(4))

23 candidate features:
['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Region', 'Order State', 'Product Name', 'Product Price', 'Shipping Mode']

Shipping mix, train vs test:
                 train    test
Shipping Mode                 
Standard Class  0.5959  0.6017
Second Class    0.1964  0.1928
First Class     0.1535  0.1533
Same Day        0.0542  0.0522
